# 🎬 Supernan AI Intern – Hindi Dubbing Pipeline
**Google Colab Notebook (Free T4 GPU)**

This notebook runs the full 7-stage pipeline on a 15-second clip of the Supernan training video.

Stages:
1. Install dependencies
2. Clone repo & download video
3. Extract clip + audio
4. Transcribe (Whisper)
5. Translate → Hindi (IndicTrans2)
6. Synthesise Hindi voice (Coqui XTTS v2)
7. Lip-sync (VideoReTalking)
8. Face restore (GFPGAN)
9. Assemble & download output

> **Runtime**: Make sure to set Runtime → T4 GPU before running.

## Cell 1 – Install System Packages

In [ ]:
# Install ffmpeg, rubberband (pitch-preserving time-stretch)
!apt-get install -y -q ffmpeg rubberband-cli
!ffmpeg -version | head -1

## Cell 2 – Install Python Dependencies

In [ ]:
!pip install -q \
    ffmpeg-python pydub colorlog \
    openai-whisper \
    deep-translator \
    TTS pyrubberband soundfile \
    facexlib basicsr gfpgan \
    huggingface_hub transformers sentencepiece sacremoses \
    opencv-python-headless

print('✓ Python packages installed')

## Cell 3 – Install IndicTrans2 (context-aware Hindi translation)

In [ ]:
!pip install -q git+https://github.com/AI4Bharat/IndicTrans2.git
print('✓ IndicTrans2 installed')

## Cell 4 – Clone VideoReTalking

In [ ]:
import os

if not os.path.exists('video-retalking'):
    !git clone https://github.com/vinthony/video-retalking.git

# Download VideoReTalking checkpoints
if not os.path.exists('video-retalking/checkpoints'):
    os.makedirs('video-retalking/checkpoints', exist_ok=True)
    from huggingface_hub import snapshot_download
    snapshot_download(
        'vinthony/video-retalking',
        local_dir='video-retalking/checkpoints',
        ignore_patterns=['*.git*'],
    )

!pip install -q -r video-retalking/requirements.txt
print('✓ VideoReTalking ready')

## Cell 5 – Clone Pipeline Repo

In [ ]:
# Replace with your actual GitHub repo URL before submitting
REPO_URL = 'https://github.com/YOUR_USERNAME/AI_Automation.git'

if not os.path.exists('AI_Automation'):
    !git clone {REPO_URL}

import sys
sys.path.insert(0, '/content/AI_Automation')
print('✓ Pipeline repo ready')

## Cell 6 – Download Source Video

You have two options:
- Option A: Upload manually via `files.upload()`
- Option B: Use gdown if you have the file ID

In [ ]:
# Option A: Upload from your computer
# from google.colab import files
# uploaded = files.upload()  # select input.mp4

# Option B: Download from Google Drive (public link)
!pip install -q gdown
FILE_ID = '1urRXU3HGjL30lXxQakqK_5rVjbH9XW3O'
!gdown --id {FILE_ID} -O /content/input.mp4

import os
assert os.path.exists('/content/input.mp4'), 'input.mp4 not found!'
print(f'✓ Video ready: {os.path.getsize("/content/input.mp4") / 1e6:.1f} MB')

## Cell 7 – Run the Pipeline

In [ ]:
os.chdir('/content/AI_Automation')

# Full GPU run: 0:15 – 0:30 segment
!python dub_video.py \
    --input /content/input.mp4 \
    --output /content/output.mp4 \
    --start 15 \
    --end 30 \
    --model base \
    --verbose

## Cell 8 – Preview & Download Output

In [ ]:
from IPython.display import Video, display
import os

output_path = '/content/output.mp4'
assert os.path.exists(output_path), 'output.mp4 not found – check pipeline logs above'
size_mb = os.path.getsize(output_path) / 1e6
print(f'Output size: {size_mb:.1f} MB')

display(Video(output_path, embed=True, width=640))

In [ ]:
# Download to your local machine
from google.colab import files
files.download('/content/output.mp4')

---
## 💡 Tips

- Switch to **large-v3** Whisper model (`--model large-v3`) for best transcription quality on A100 Colab runtime
- Add `--batch` flag if processing a video longer than 5 minutes to avoid OOM
- Intermediate files are saved in `tmp/` with per-stage checkpoints; re-run the cell to resume from the last successful stage
- Replace `REPO_URL` in Cell 5 with your actual GitHub repo before submitting